# 🧠 geDIG: Graph-based Emergent Discovery of Insight Graphs

**Multi-hop reasoning for knowledge discovery in maze navigation**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miyauchikazuyoshi/InsightSpike-AI/blob/main/experiments/maze/colab_maze_experiment.ipynb)

---

## What is geDIG?

geDIG is an algorithm that discovers **structural insights** by evaluating multi-hop graph expansions. In maze navigation:

- **g₀**: Score at hop 0 (immediate neighbors only)
- **g_min**: Best score across all hops (multi-hop reasoning)
- **k★**: Optimal hop depth where g_min is achieved
- **ΔSP**: Shortest path improvement through graph shortcuts

When **g_min < g₀**, multi-hop reasoning provides better insights than greedy local decisions.

---

## Quick Start

1. **Quick Demo** → Run in ~2 min, see results immediately
2. **Full Experiment** → Larger scale with statistical significance

In [ ]:
# @title 🚀 Setup (Run this first)
import os

# Clone repository
REPO_DIR = '/content/InsightSpike-AI'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/miyauchikazuyoshi/InsightSpike-AI.git {REPO_DIR}
os.chdir(REPO_DIR)

# Install dependencies
!pip install -q networkx numpy matplotlib ipywidgets

print('✅ Setup complete!')

---
## 🎮 Quick Demo (25×25 Maze)

**3 seeds × 200 steps** → Results in ~2 minutes

In [ ]:
# @title ▶️ Run Quick Demo
import os
import time

os.chdir('/content/InsightSpike-AI')
DEMO_DIR = '/content/demo_results'
!mkdir -p {DEMO_DIR}

print('🏃 Running 25×25 maze with 3 seeds...')
print('=' * 50)
start_time = time.time()

!PYTHONPATH=src python experiments/maze/run_experiment_query.py \
  --maze-size 25 --max-steps 200 --seeds 3 --workers 2 \
  --max-hops 10 --sp-pair-samples 128 --sp-cand-topk 12 \
  --lambda-weight 1.0 --sp-beta 1.0 --linkset-mode \
  --sp-scope union --sp-hop-expand 3 \
  --theta-ag -1.0 --theta-dg 0.15 \
  --gh-mode greedy --eval-all-hops \
  --output {DEMO_DIR}/summary.json \
  --step-log {DEMO_DIR}/steps.json

elapsed = time.time() - start_time
print('=' * 50)
print(f'✅ Demo completed in {elapsed:.1f} seconds')

In [ ]:
# @title 📊 Demo Results - Summary Cards
import json
from IPython.display import HTML, display

with open('/content/demo_results/summary.json', 'r') as f:
    demo_summary = json.load(f)

demo_runs = demo_summary.get('runs', [])
demo_agg = demo_summary.get('aggregated', {})

# Beautiful summary cards
cards_html = '''
<style>
    .demo-container { display: flex; flex-wrap: wrap; gap: 16px; margin: 20px 0; justify-content: center; }
    .demo-card {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        border-radius: 16px; padding: 20px 28px; min-width: 160px; text-align: center;
        box-shadow: 0 8px 25px rgba(102, 126, 234, 0.4);
        transition: transform 0.2s;
    }
    .demo-card:hover { transform: translateY(-4px); }
    .demo-title { color: rgba(255,255,255,0.85); font-size: 12px; text-transform: uppercase; letter-spacing: 1.5px; }
    .demo-value { color: white; font-size: 36px; font-weight: bold; margin-top: 8px; }
    .demo-subtitle { color: rgba(255,255,255,0.7); font-size: 11px; margin-top: 4px; }
</style>
<h2 style="text-align:center; color:#4a5568;">🎯 geDIG Demo Results</h2>
<div class="demo-container">
'''

metrics = [
    ('Success Rate', f"{demo_agg.get('success_rate', 0):.0%}", 'Goal reached'),
    ('Avg Steps', f"{demo_agg.get('avg_steps', 0):.0f}", 'To reach goal'),
    ('g_min Mean', f"{demo_agg.get('gmin_mean', 0):.3f}", 'Lower is better'),
    ('Best Hop k★', f"{demo_agg.get('best_hop_mean', 0):.1f}", 'Multi-hop depth'),
    ('ΔSP Gain', f"{demo_agg.get('avg_delta_sp_min', 0):.3f}", 'Path improvement'),
]

for title, value, subtitle in metrics:
    cards_html += f'''
    <div class="demo-card">
        <div class="demo-title">{title}</div>
        <div class="demo-value">{value}</div>
        <div class="demo-subtitle">{subtitle}</div>
    </div>'''

cards_html += '</div>'
display(HTML(cards_html))

In [ ]:
# @title 📈 Demo Results - Temporal Metrics
import matplotlib.pyplot as plt
import numpy as np

# Plot all seeds
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ['#667eea', '#f56565', '#48bb78']

for i, run in enumerate(demo_runs[:3]):
    ax = axes[i]
    g0 = run.get('g0_series', [])
    gmin = run.get('gmin_series', [])
    steps = range(len(g0))
    
    ax.plot(steps, g0, label='g₀ (hop=0)', color='#3182ce', linewidth=2, alpha=0.8)
    ax.plot(steps, gmin, label='g_min (best)', color='#e53e3e', linewidth=2, alpha=0.8)
    ax.fill_between(steps, g0, gmin, alpha=0.2, color='#48bb78', label='Multi-hop gain')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.4)
    
    status = '✓ Success' if run.get('success') else '✗ Timeout'
    ax.set_title(f'Seed {run.get("seed", i)} ({status})', fontsize=12, fontweight='bold')
    ax.set_xlabel('Step')
    ax.set_ylabel('Score')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('geDIG Score Evolution: g₀ vs g_min', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Explanation
print('\n📖 Interpretation:')
print('  • Blue (g₀): Score using only immediate neighbors')
print('  • Red (g_min): Best score across all hop depths')
print('  • Green area: Benefit from multi-hop reasoning')
print('  • When g_min < g₀, multi-hop provides better insights!')

In [ ]:
# @title 🗺️ Demo Results - Maze Visualization
import matplotlib.pyplot as plt
import numpy as np
import json

with open('/content/demo_results/steps.json', 'r') as f:
    demo_steps = json.load(f)

if 'maze_snapshots' in demo_steps and demo_steps['maze_snapshots']:
    maze = demo_steps['maze_snapshots'][0]
    layout = np.array(maze['layout'])
    start = maze['start_pos']
    goal = maze['goal_pos']
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Custom colormap
    cmap = plt.cm.colors.ListedColormap(['#f7fafc', '#2d3748', '#4299e1', '#48bb78'])
    ax.imshow(layout, cmap=cmap, vmin=0, vmax=3)
    
    # Mark start and goal
    ax.scatter(start[1], start[0], c='#3182ce', s=300, marker='s', label='Start', zorder=10, edgecolors='white', linewidths=2)
    ax.scatter(goal[1], goal[0], c='#38a169', s=400, marker='*', label='Goal', zorder=10, edgecolors='white', linewidths=2)
    
    ax.set_title('25×25 Maze (Seed 0)', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Add border
    for spine in ax.spines.values():
        spine.set_edgecolor('#667eea')
        spine.set_linewidth(3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# @title 📊 Demo Results - k★ Distribution
import matplotlib.pyplot as plt
import numpy as np

# Collect all k★ values
all_kstar = []
for run in demo_runs:
    all_kstar.extend(run.get('multihop_best_hop', []))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# k★ histogram
ax1 = axes[0]
if all_kstar:
    unique_k = sorted(set(all_kstar))
    counts = [all_kstar.count(k) for k in unique_k]
    colors = ['#667eea' if k == 0 else '#48bb78' for k in unique_k]
    bars = ax1.bar(unique_k, counts, color=colors, edgecolor='white', linewidth=1.5)
    
    # Add percentage labels
    total = sum(counts)
    for bar, count in zip(bars, counts):
        pct = 100 * count / total
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                f'{pct:.0f}%', ha='center', fontsize=10, fontweight='bold')

ax1.set_xlabel('Best Hop (k★)', fontsize=11)
ax1.set_ylabel('Count', fontsize=11)
ax1.set_title('k★ Distribution', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Multi-hop usage pie
ax2 = axes[1]
hop0_count = all_kstar.count(0)
multihop_count = len(all_kstar) - hop0_count
ax2.pie([hop0_count, multihop_count], 
        labels=['Hop 0 (local)', 'Hop ≥1 (multi-hop)'],
        colors=['#667eea', '#48bb78'],
        autopct='%1.1f%%', startangle=90,
        explode=(0, 0.05), shadow=True,
        textprops={'fontsize': 11, 'fontweight': 'bold'})
ax2.set_title('Multi-hop Usage', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n📖 Multi-hop was beneficial in {100*multihop_count/len(all_kstar):.1f}% of decisions!')

---
## 🔬 Full Experiment

For statistically significant results, run with more seeds.

In [ ]:
# @title 1. Mount Google Drive (for saving results)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 2. Experiment Parameters { run: "auto" }

MAZE_SIZE = 25  # @param {type:"integer"}
MAX_STEPS = 500  # @param {type:"integer"}
SEEDS = 60  # @param {type:"integer"}
WORKERS = 2  # @param {type:"integer"}
GH_MODE = "greedy"  # @param ["greedy", "radius"]

OUTPUT_DIR = f'/content/drive/MyDrive/maze_results/{MAZE_SIZE}x{MAZE_SIZE}_s{MAX_STEPS}_seed{SEEDS}'
!mkdir -p {OUTPUT_DIR}
print(f'📁 Output: {OUTPUT_DIR}')

In [ ]:
# @title 3. Run Full Experiment
import os
os.chdir('/content/InsightSpike-AI')

!PYTHONPATH=src python experiments/maze/run_experiment_query.py \
  --maze-size {MAZE_SIZE} --max-steps {MAX_STEPS} --seeds {SEEDS} --workers {WORKERS} \
  --max-hops 10 --sp-pair-samples 128 --sp-cand-topk 12 \
  --lambda-weight 1.0 --sp-beta 1.0 --linkset-mode \
  --sp-scope union --sp-hop-expand 3 \
  --theta-ag -1.0 --theta-dg 0.15 \
  --gh-mode {GH_MODE} --eval-all-hops \
  --output {OUTPUT_DIR}/summary.json \
  --step-log {OUTPUT_DIR}/steps.json

In [ ]:
# @title 4. Check Progress
import glob, json, os

for f in glob.glob(f'{OUTPUT_DIR}/*.incremental.jsonl'):
    with open(f) as fp:
        lines = fp.readlines()
    success = sum(1 for l in lines if json.loads(l).get('summary', {}).get('success', False))
    print(f'{os.path.basename(f)}: {len(lines)} done, {success} success ({100*success/max(1,len(lines)):.0f}%)')

In [ ]:
# @title 5. Load & Visualize Full Results
import json
import matplotlib.pyplot as plt
import numpy as np

with open(f'{OUTPUT_DIR}/summary.json') as f:
    summary = json.load(f)

runs = summary.get('runs', [])
agg = summary.get('aggregated', {})

# Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Steps
ax1 = axes[0, 0]
steps = [r['steps'] for r in runs]
success = [r['success'] for r in runs]
colors = ['#48bb78' if s else '#f56565' for s in success]
ax1.bar(range(len(steps)), steps, color=colors)
ax1.axhline(np.mean(steps), color='blue', linestyle='--')
ax1.set_title(f'Steps per Seed (Mean: {np.mean(steps):.0f})')

# gmin
ax2 = axes[0, 1]
gmin_final = [r['gmin_series'][-1] for r in runs if r['gmin_series']]
ax2.hist(gmin_final, bins=20, color='#e53e3e', alpha=0.7, edgecolor='black')
ax2.axvline(np.mean(gmin_final), color='blue', linestyle='--')
ax2.set_title(f'g_min Distribution (Mean: {np.mean(gmin_final):.3f})')

# k★
ax3 = axes[1, 0]
all_k = []
for r in runs:
    all_k.extend(r.get('multihop_best_hop', []))
unique_k = sorted(set(all_k))
ax3.bar(unique_k, [all_k.count(k) for k in unique_k], color='#48bb78')
ax3.set_title('k★ Distribution')

# Success
ax4 = axes[1, 1]
ax4.pie([sum(success), len(success)-sum(success)], labels=['Success', 'Fail'],
        colors=['#48bb78', '#f56565'], autopct='%1.1f%%')
ax4.set_title(f'Success Rate: {sum(success)}/{len(success)}')

plt.tight_layout()
plt.show()

In [ ]:
# @title 6. Generate HTML Report
!python experiments/maze/build_reports.py \
  --summary {OUTPUT_DIR}/summary.json \
  --steps {OUTPUT_DIR}/steps.json \
  --out {OUTPUT_DIR}/interactive.html

print(f'\n✅ Report: {OUTPUT_DIR}/interactive.html')

---
## 🚀 51×51 Paper Experiment (Recommended)

Use the dedicated script for reliable large-scale runs with:
- **40 seeds, 1500 steps**
- Auto-resume on timeout
- Progress tracking
- Optimized parameters

In [ ]:
# @title ▶️ Run 51×51 Paper Experiment
import os
os.chdir('/content/InsightSpike-AI')

# Mount Drive for persistent storage
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

OUTPUT_51 = '/content/drive/MyDrive/maze_results/51x51_paper'
!mkdir -p {OUTPUT_51}

# Run using the paper experiment script
!PYTHONPATH=src python experiments/maze/run_paper_51x51.py \
    --output-dir {OUTPUT_51} \
    --workers 2

print(f'\n📁 Results saved to: {OUTPUT_51}')

In [ ]:
# @title 📊 Check Progress / Resume
import os
os.chdir('/content/InsightSpike-AI')

OUTPUT_51 = '/content/drive/MyDrive/maze_results/51x51_paper'

# Check current status
!PYTHONPATH=src python experiments/maze/run_paper_51x51.py --status --output-dir {OUTPUT_51}

# To resume from where it stopped, uncomment:
# !PYTHONPATH=src python experiments/maze/run_paper_51x51.py --resume --output-dir {OUTPUT_51} --workers 2

---
## 💡 Tips

| Feature | Benefit |
|---------|--------|
| **Colab** | No local setup, save battery |
| **Google Drive** | Results persist after session |
| **Incremental save** | Resume after timeout |
| **Workers=2** | Optimal for free Colab |